# 1. Imports & Dependencies

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Day3.x module bootstrap (ref-macro)
import sys
from pathlib import Path

_local_src = str(Path.cwd() / "src")
if _local_src not in sys.path:
    sys.path.insert(0, _local_src)

from lakehouse.ref_parser import parse_ecb_fx_xml, parse_fred_observations
from lakehouse.macro_rules import is_valid_ecb_fx_record, is_valid_fred_record

print("[day3.x] imported ref_parser + macro_rules")

ecb_item_schema = T.StructType([
    T.StructField("rate_date", T.StringType(), True),
    T.StructField("base_ccy", T.StringType(), True),
    T.StructField("quote_ccy", T.StringType(), True),
    T.StructField("fx_rate", T.DoubleType(), True),
])

fred_item_schema = T.StructType([
    T.StructField("obs_date", T.StringType(), True),
    T.StructField("value", T.DoubleType(), True),
])

@F.udf(T.ArrayType(ecb_item_schema))
def parse_ecb_xml_udf(raw_xml, base_ccy):
    rows = parse_ecb_fx_xml(raw_xml, base_ccy=(base_ccy or "EUR"))
    return [r for r in rows if is_valid_ecb_fx_record(r)]

@F.udf(T.ArrayType(fred_item_schema))
def parse_fred_json_udf(raw_json):
    rows = parse_fred_observations(raw_json)
    return [r for r in rows if is_valid_fred_record(r)]

# 2. Widget Definitions

In [0]:
# Catalog & Schema setup
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("bronze_schema", "bronze_ref")
dbutils.widgets.text("silver_schema", "silver_ref")

# Table Names
dbutils.widgets.text("ecb_bronze_table", "ecb_fx_raw")
dbutils.widgets.text("fred_bronze_table", "fred_series_raw")
dbutils.widgets.text("ecb_silver_table", "ecb_fx_daily")
dbutils.widgets.text("fred_silver_table", "fred_series_daily")

# Filters (Optional)
dbutils.widgets.text("ecb_quote_ccy_filter", "USD,GBP,CNY") 
dbutils.widgets.text("fred_series_filter", "DFF,CPIAUCSL")

# 3. Configuration Parsing

In [0]:
CATALOG = dbutils.widgets.get("catalog").strip()
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema").strip()
SILVER_SCHEMA = dbutils.widgets.get("silver_schema").strip()

ECB_BRONZE_TBL  = f"{CATALOG}.{BRONZE_SCHEMA}.{dbutils.widgets.get('ecb_bronze_table').strip()}"
FRED_BRONZE_TBL = f"{CATALOG}.{BRONZE_SCHEMA}.{dbutils.widgets.get('fred_bronze_table').strip()}"
ECB_SILVER_TBL  = f"{CATALOG}.{SILVER_SCHEMA}.{dbutils.widgets.get('ecb_silver_table').strip()}"
FRED_SILVER_TBL = f"{CATALOG}.{SILVER_SCHEMA}.{dbutils.widgets.get('fred_silver_table').strip()}"

# Parse filters into lists
ecb_filter = [s.strip().upper() for s in dbutils.widgets.get("ecb_quote_ccy_filter").split(",") if s.strip()]
fred_filter = [s.strip().upper() for s in dbutils.widgets.get("fred_series_filter").split(",") if s.strip()]

print(f"[INFO] Config Ready:")
print(f"       ECB:  {ECB_BRONZE_TBL} -> {ECB_SILVER_TBL}")
print(f"       FRED: {FRED_BRONZE_TBL} -> {FRED_SILVER_TBL}")

# 4. ECB Pipeline (Logic + Write)

In [0]:
%skip
print("[INFO] Processing ECB FX Data...")

# 1. DDL: Ensure Target Exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ECB_SILVER_TBL} (
  rate_date      DATE,
  base_ccy       STRING,
  quote_ccy      STRING,
  fx_rate        DOUBLE,
  ingestion_ts   TIMESTAMP
)
USING DELTA
PARTITIONED BY (rate_date)
""")

# 2. Parsing Logic (Regex Strategy)
chunks = (spark.table(ECB_BRONZE_TBL)
  .select("base_ccy", "raw_json", "ingestion_ts")
  .withColumn("block", F.explode(F.split(F.col("raw_json"), 'time="')))
  .where(F.col("block").rlike(r"^\d{4}-\d{2}-\d{2}")) 
  .withColumn("rate_date", F.to_date(F.regexp_extract(F.col("block"), r"^(\d{4}-\d{2}-\d{2})", 1)))
  # Split currency blocks
  .withColumn("pair", F.explode(F.split(F.col("block"), '<Cube currency="')))
  .where(F.col("pair").rlike(r'^[A-Z]{3}" rate="'))
  # Extract values
  .withColumn("quote_ccy", F.regexp_extract(F.col("pair"), r'^([A-Z]{3})"', 1))
  .withColumn("fx_rate", F.regexp_extract(F.col("pair"), r'rate="([0-9.]+)"', 1).cast("double"))
  .select("rate_date", "base_ccy", "quote_ccy", "fx_rate", "ingestion_ts")
)

# 3. Apply Filters
if ecb_filter:
    chunks = chunks.where(F.col("quote_ccy").isin(ecb_filter))

# 4. Deduplicate (Keep latest ingestion)
wkey = Window.partitionBy("rate_date", "base_ccy", "quote_ccy").orderBy(F.col("ingestion_ts").desc())
ecb_final = chunks.withColumn("rn", F.row_number().over(wkey)).where(F.col("rn") == 1).drop("rn")

# 5. Merge (Upsert)
ecb_final.createOrReplaceTempView("stg_ecb_fx")
spark.sql(f"""
MERGE INTO {ECB_SILVER_TBL} AS tgt
USING stg_ecb_fx AS src
ON  tgt.rate_date = src.rate_date AND tgt.base_ccy = src.base_ccy AND tgt.quote_ccy = src.quote_ccy
WHEN MATCHED THEN UPDATE SET tgt.fx_rate = src.fx_rate, tgt.ingestion_ts = src.ingestion_ts
WHEN NOT MATCHED THEN INSERT *
""")

print(f"[SUCCESS] ECB Merge completed into {ECB_SILVER_TBL}")

In [0]:
print("[INFO] Processing ECB FX Data...")

# 1. DDL: Ensure Target Exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ECB_SILVER_TBL} (
  rate_date      DATE,
  base_ccy       STRING,
  quote_ccy      STRING,
  fx_rate        DOUBLE,
  ingestion_ts   TIMESTAMP
)
USING DELTA
PARTITIONED BY (rate_date)
""")

# 2. Parsing Logic (moduleized parser + rules)
chunks = (spark.table(ECB_BRONZE_TBL)
  .select("base_ccy", "raw_json", "ingestion_ts")
  .withColumn("parsed", parse_ecb_xml_udf(F.col("raw_json"), F.col("base_ccy")))
  .withColumn("item", F.explode_outer(F.col("parsed")))
  .select(
      F.to_date(F.col("item.rate_date")).alias("rate_date"),
      F.col("item.base_ccy").alias("base_ccy"),
      F.col("item.quote_ccy").alias("quote_ccy"),
      F.col("item.fx_rate").cast("double").alias("fx_rate"),
      F.col("ingestion_ts")
  )
  .filter(F.col("rate_date").isNotNull())
  .filter(F.col("fx_rate").isNotNull())
)

# 3. Apply Filters
if ecb_filter:
    chunks = chunks.where(F.col("quote_ccy").isin(ecb_filter))

# 4. Deduplicate (Keep latest ingestion)
wkey = Window.partitionBy("rate_date", "base_ccy", "quote_ccy").orderBy(F.col("ingestion_ts").desc())
ecb_final = chunks.withColumn("rn", F.row_number().over(wkey)).where(F.col("rn") == 1).drop("rn")

# 5. Merge (Upsert)
ecb_final.createOrReplaceTempView("stg_ecb_fx")
print(f"[INFO] Merging {ecb_final.count()} rows into Silver table...")

spark.sql(f"""
MERGE INTO {ECB_SILVER_TBL} AS tgt
USING stg_ecb_fx AS src
ON  tgt.rate_date = src.rate_date AND tgt.base_ccy = src.base_ccy AND tgt.quote_ccy = src.quote_ccy
WHEN MATCHED THEN UPDATE SET tgt.fx_rate = src.fx_rate, tgt.ingestion_ts = src.ingestion_ts
WHEN NOT MATCHED THEN INSERT *
""")

print(f"[SUCCESS] ECB Merge completed into {ECB_SILVER_TBL}")

# 5. FRED Pipeline (Logic + Write)

In [0]:
print("[INFO] Processing FRED Series Data...")

# 1. DDL
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FRED_SILVER_TBL} (
  series_id     STRING,
  obs_date      DATE,
  value         DOUBLE,
  ingestion_ts  TIMESTAMP
)
USING DELTA
PARTITIONED BY (series_id) 
""")
# Note: Partitioning by series_id is usually better for FRED data query patterns

# 2. Parsing Logic (moduleized parser + rules)
fred_parsed = (spark.table(FRED_BRONZE_TBL)
  .select("series_id", "raw_json", "ingestion_ts")
  .withColumn("parsed", parse_fred_json_udf(F.col("raw_json")))
  .withColumn("obs", F.explode_outer(F.col("parsed")))
  .select(
      F.col("series_id"),
      F.to_date(F.col("obs.obs_date")).alias("obs_date"),
      F.col("obs.value").cast("double").alias("value"),
      F.col("ingestion_ts")
  )
  .filter(F.col("obs_date").isNotNull())
  .filter(F.col("value").isNotNull())
)

# 3. Apply Filters
if fred_filter:
    fred_parsed = fred_parsed.where(F.col("series_id").isin(fred_filter))

# 4. Deduplicate: keep latest ingestion for (series_id, obs_date)
wkey_fred = Window.partitionBy("series_id", "obs_date").orderBy(F.col("ingestion_ts").desc())
fred_final = fred_parsed.withColumn("rn", F.row_number().over(wkey_fred)).where(F.col("rn") == 1).drop("rn")

# 5. Merge (Upsert)
fred_final.createOrReplaceTempView("stg_fred_series")
spark.sql(f"""
MERGE INTO {FRED_SILVER_TBL} AS tgt
USING stg_fred_series AS src
ON  tgt.series_id = src.series_id AND tgt.obs_date = src.obs_date
WHEN MATCHED THEN UPDATE SET tgt.value = src.value, tgt.ingestion_ts = src.ingestion_ts
WHEN NOT MATCHED THEN INSERT *
""")

print(f"[SUCCESS] FRED Merge completed into {FRED_SILVER_TBL}")

# 6. Validation

In [0]:
# Quick check of the results
print("--- ECB Sample ---")
display(spark.table(ECB_SILVER_TBL).orderBy(F.col("rate_date").desc()).limit(5))

print("--- FRED Sample ---")
display(spark.table(FRED_SILVER_TBL).orderBy(F.col("obs_date").desc()).limit(100))

In [0]:
# Debug Cell 1: Check Bronze Content
# 目的：查看 raw_json 列的内容，确认 XML 是否下载成功，以及引号格式是单引号(')还是双引号(")
bronze_df = spark.table(ECB_BRONZE_TBL)
print(f"Bronze Row Count: {bronze_df.count()}")

# 显示第一行的 raw_json (XML string)
raw_record = bronze_df.select("raw_json").first()
if raw_record:
    print("--- Raw XML Sample (First 500 chars) ---")
    print(raw_record["raw_json"][:500])
else:
    print("[ERROR] Bronze table is empty!")